# The Monty Hall Problem

## Setup

You are a contestant on a game show. Three doors stand before you:
- Behind **one door** is a brand-new car 🚗
- Behind the **other two doors** are goats 🐐

The game works like this:
1. You pick a door (say, Door 1).
2. The host — who *knows* where the car is — opens one of the other two doors, always revealing a goat.
3. You are given a choice: **stick with your original door**, or **switch** to the remaining closed door.

**The question:** Should you switch?

Most people say it doesn't matter — two doors remain, one car, so it must be 50/50… right?

## The Surprising Answer

The correct answer is: **you should always switch.**

Switching wins approximately **2 out of 3 times**. Staying wins only **1 out of 3 times**.

This feels wrong. After the host reveals a goat, two closed doors remain. Surely it's 50/50?

This is one of the most famous counterintuitive results in probability theory. When Marilyn vos Savant published the correct answer in 1990 — in a column read by 10 million people — thousands of readers wrote in to disagree, including tenured mathematicians.

We'll settle it empirically: simulate the game thousands of times and count.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## Simulating a Single Game

We'll model every step of the game explicitly — no shortcuts:
1. Place the car behind a random door.
2. The contestant picks a random door.
3. The host opens a door that is **neither the contestant's pick nor the car**, revealing a goat.
4. The contestant either stays or switches to the remaining closed door.

This explicit simulation makes the mechanics transparent and easy to verify.

In [ ]:
def play_one_game(switch, rng):
    '''Simulate one Monty Hall game; return True if contestant wins.

    switch: True to switch doors after the host's reveal.
    rng: numpy random generator (pass a seeded one for reproducibility).
    '''
    doors = [0, 1, 2]
    car  = rng.integers(0, 3)        # car placed behind random door
    pick = rng.integers(0, 3)        # contestant picks random door

    # host opens a door that is neither the pick nor the car
    valid_reveals = [d for d in doors if d != pick and d != car]
    host_opens = rng.choice(valid_reveals)

    if switch:
        # move to the one remaining closed door
        final = [d for d in doors if d != pick and d != host_opens][0]
    else:
        final = pick

    return final == car

In [ ]:
# Show both strategies on the same randomly generated scenario
rng_demo = np.random.default_rng(seed=7)
doors  = [0, 1, 2]
labels = ['Door 1', 'Door 2', 'Door 3']

car  = rng_demo.integers(0, 3)
pick = rng_demo.integers(0, 3)
valid_reveals = [d for d in doors if d != pick and d != car]
host_opens = rng_demo.choice(valid_reveals)

stay_choice   = pick
switch_choice = [d for d in doors if d != pick and d != host_opens][0]

stay_result   = 'WIN  🚗' if stay_choice   == car else 'LOSE 🐐'
switch_result = 'WIN  🚗' if switch_choice == car else 'LOSE 🐐'

print(f'Car is behind : {labels[car]}')
print(f'You pick      : {labels[pick]}')
print(f'Host reveals  : {labels[host_opens]}  (it is a goat)')
print()
print(f'STAY   → keep {labels[stay_choice]}     → {stay_result}')
print(f'SWITCH → move to {labels[switch_choice]} → {switch_result}')
print()
print('Run this cell a few times to see different scenarios.')

## Running Many Games

One game is just anecdote — we got lucky or unlucky. Let's run **10,000 games** with each strategy and see where the win rates land.

In [ ]:
def run_trials(n_trials, switch, rng):
    '''Run n_trials games with one strategy; return number of wins.'''
    return sum(play_one_game(switch, rng) for _ in range(n_trials))

N_TRIALS = 10_000
rng = np.random.default_rng()

stay_wins   = run_trials(N_TRIALS, switch=False, rng=rng)
switch_wins = run_trials(N_TRIALS, switch=True,  rng=rng)

stay_rate   = stay_wins   / N_TRIALS
switch_rate = switch_wins / N_TRIALS

print(f'Stay  : {stay_wins:,} wins / {N_TRIALS:,} = {stay_rate:.3f}  '
      f'(theory: {1/3:.3f})')
print(f'Switch: {switch_wins:,} wins / {N_TRIALS:,} = {switch_rate:.3f}  '
      f'(theory: {2/3:.3f})')

## Visualising the Results

Two charts:
- **Left** — aggregate win rates vs. the theoretical values (dashed lines)
- **Right** — cumulative win rates over trials, showing convergence

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))

colors = ['#4c72b0', '#dd8452']
bars = ax.bar(['Stay', 'Switch'], [stay_rate, switch_rate],
              color=colors, width=0.5, zorder=2)
ax.axhline(1/3, color='#4c72b0', linestyle='--', linewidth=1.2,
           label='Theoretical (1/3)', zorder=1)
ax.axhline(2/3, color='#dd8452', linestyle='--', linewidth=1.2,
           label='Theoretical (2/3)', zorder=1)
for bar, rate in zip(bars, [stay_rate, switch_rate]):
    ax.text(bar.get_x() + bar.get_width() / 2, rate + 0.01,
            f'{rate:.3f}', ha='center', va='bottom', fontsize=11)
ax.set_ylim(0, 1)
ax.set_ylabel('Win Rate')
ax.set_title(f'Win Rate by Strategy  (N = {N_TRIALS:,})')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Vectorised convergence — equivalent to run_trials but fast enough for large N.
# Key insight: staying wins iff the original pick == car (no host simulation needed).
rng_conv = np.random.default_rng()
car_arr  = rng_conv.integers(0, 3, size=N_TRIALS)
pick_arr = rng_conv.integers(0, 3, size=N_TRIALS)

stay_results = (pick_arr == car_arr).astype(float)   # 1 where stay wins
counts       = np.arange(1, N_TRIALS + 1)
stay_curve   = np.cumsum(stay_results) / counts       # running win rate
switch_curve = 1.0 - stay_curve                       # always the complement

trials = np.arange(1, N_TRIALS + 1)
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(trials, stay_curve,   color='#4c72b0', linewidth=1.2, label='Stay')
ax.plot(trials, switch_curve, color='#dd8452', linewidth=1.2, label='Switch')
ax.axhline(1/3, color='#4c72b0', linestyle='--', linewidth=1.0, label='Theoretical (1/3)')
ax.axhline(2/3, color='#dd8452', linestyle='--', linewidth=1.0, label='Theoretical (2/3)')
ax.set_xscale('log')
ax.set_ylim(0, 1)
ax.set_xlabel('Number of Trials')
ax.set_ylabel('Cumulative Win Rate')
ax.set_title('Convergence to Theoretical Probability')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Why Does Switching Win 2/3 of the Time?

### The direct argument

When you first pick a door, you have a **1/3 chance of being right** (the car is behind your door) and a **2/3 chance of being wrong** (the car is behind one of the other two doors).

The host now reveals a goat behind one of the doors you *didn't* pick. This doesn't change the probability that your original pick was correct — it's still 1/3. But the 2/3 probability that the car is "over there" is now concentrated entirely on the *one* remaining door.

| Strategy | Wins when… | Probability |
|---|---|---|
| **Stay** | Your original pick was the car | 1/3 |
| **Switch** | Your original pick was *not* the car | 2/3 |

**Switching wins exactly when your first guess was wrong** — and you're wrong 2 out of 3 times.

### The shortcut in the convergence plot

Notice the vectorised code above uses:

```python
stay_results = (pick == car)    # staying wins iff original pick == car
switch_curve = 1 - stay_curve   # switching wins the complement
```

This is correct because once the host reveals a goat, the switcher wins if and only if the original pick was *not* the car — the host's specific door choice is irrelevant to that fact.

### A Bayesian view (optional)

Before the reveal, P(car = your door) = 1/3. The host's action is *guaranteed* regardless of whether your door hides the car or a goat — it carries no information about your door. So your posterior stays at 1/3, and the other closed door absorbs the remaining 2/3.

## Try It Yourself

Some experiments to run:

- **Change `N_TRIALS`** — try 100, 1 000, 100 000. Watch how the win rates and convergence curves behave.
- **Fix the seed** — set `rng = np.random.default_rng(seed=42)` to get reproducible results you can share with others.
- **Four doors** — modify `play_one_game` to use 4 doors (host still reveals exactly 1 goat). What switching advantage do you expect? Does the simulation confirm it?
- **Stubborn host** — make the host *always* open the lowest-numbered valid door (no randomness among valid reveals). Does the aggregate win rate change? It shouldn't — but verify it!